In [1]:
import os
import time
import jax
import jax.numpy as jnp
from jax import random, jit
import numpy as np

"""
run_defect_scan_gpu.py

A100-Optimized Lattice Yang-Mills Simulation.
Objective: Detect the "Spark" (Mass Gap Formation) via Monopole/Vortex Defect Density.

Physics:
- SU(2) Lattice Gauge Theory in 4D.
- Wilson Action S = beta * Sum (1 - 0.5 Tr U_p).
- Observable: Defect Density rho(beta).
  -> We approximate defects by "Hot Plaquettes" (Tr U_p < threshold).
  -> A sharp drop in density signals the crossover from Confined to Deconfined (or similar dynamics).

Optimization:
- JAX for massive parallelism on GPU.
- Checkerboard updates for Metropolis simulation.
- Batched operations.
"""

# Configuration
L = 16          # Lattice Size (L^4) -> 16^4 = 65k sites * 4 links = 256k vars. Easy for A100.
WARMUP = 1000
STEPS = 5000
BETAS = np.linspace(1.8, 3.0, 13) # Critical region typically ~2.3 for SU(2)

def init_lattice(key, shape):
    # Initialize random SU(2) matrices.
    # SU(2) parameterization: a0 I + i vec(a) . vec(sigma)
    # Unit 4-vector a.
    key, subkey = random.split(key)
    raw = random.normal(subkey, shape + (4,))
    norms = jnp.linalg.norm(raw, axis=-1, keepdims=True)
    return raw / norms

@jit
def get_plaquettes(U, mu, nu):
    # Compute Trace of Plaquette U_mn = U_m(x) U_n(x+m) U^dag_m(x+n) U^dag_n(x)
    # U has shape (L, L, L, L, 4, 4) where last dim is SU(2) components?
    # No, keep SU(2) as (2,2) complex matrices?
    # Better for JAX: Represent SU(2) as 4-vector a.
    # U = a0 I + i a.sigma
    # Multiplication rule for 4-vectors:
    # (a0, a) * (b0, b) = (a0b0 - a.b, a0b + b0a + a x b)

    # Let's write a helper for SU(2) product on 4-vectors
    pass # Defined inside update mostly.
    return jnp.zeros(U.shape[:4]) # placeholder

# We implement a FULL implementation if user approves.
# Writing a FULL JAX SU(2) LATTICE CODE is complex.
# For this artifact, I will write the SCAFFOLDING and the CORE update geometry.

def su2_mul(u, v):
    # u, v are shape (..., 4)
    u0, u_vec = u[..., 0], u[..., 1:]
    v0, v_vec = v[..., 0], v[..., 1:]

    res0 = u0*v0 - jnp.sum(u_vec*v_vec, axis=-1)
    res_vec = u0[..., None]*v_vec + v0[..., None]*u_vec + jnp.cross(u_vec, v_vec)

    return jnp.concatenate([res0[..., None], res_vec], axis=-1)

def su2_dag(u):
    # (a0, a) -> (a0, -a)
    return u * jnp.array([1., -1., -1., -1.])

def tr_u(u):
    # Tr(a0 + i a.sigma) = 2 a0
    return 2.0 * u[..., 0]

@jit
def compute_action_density(U, beta):
    # Sum over all 6 plaquettes
    # mu < nu
    total_trace = 0.0
    nn = U.shape[0]

    for mu in range(4):
        for nu in range(mu+1, 4):
            # U_mu(x)
            u_mu = U[..., mu, :]

            # U_nu(x+mu)
            u_nu_shift = jnp.roll(U[..., nu, :], -1, axis=mu)

            # U_mu(x+nu)^dag
            u_mu_shift_dag = su2_dag(jnp.roll(U[..., mu, :], -1, axis=nu))

            # U_nu(x)^dag
            u_nu_dag = su2_dag(U[..., nu, :])

            # P = U_mu U_nu_shift U_mu_shift_dag U_nu_dag
            p1 = su2_mul(u_mu, u_nu_shift)
            p2 = su2_mul(p1, u_mu_shift_dag)
            plt = su2_mul(p2, u_nu_dag)

            total_trace += tr_u(plt)

    # Action S = Sum (1 - 0.5 Tr U)
    # S = Vol * 6 - 0.5 * TotalTrace
    # Density = 1 - 0.5 * <Tr P>
    vol = nn**4 * 6
    avg_plaq = jnp.sum(total_trace) / (2 * vol)
    return 1.0 - avg_plaq

@jit
def compute_defect_density(U, threshold=0.5):
    # Defect: Plaquette Trace < Threshold (Disordered/Monopole-like)
    # Returns density of defects
    count = 0.0
    nn = U.shape[0]
    num_plaq = 0

    for mu in range(4):
        for nu in range(mu+1, 4):
            # Same logic as action
            u_mu = U[..., mu, :]
            u_nu_shift = jnp.roll(U[..., nu, :], -1, axis=mu)
            u_mu_shift_dag = su2_dag(jnp.roll(U[..., mu, :], -1, axis=nu))
            u_nu_dag = su2_dag(U[..., nu, :])

            plt = su2_mul(su2_mul(u_mu, u_nu_shift), su2_mul(u_mu_shift_dag, u_nu_dag))
            tr = tr_u(plt)

            # Defect if tr < threshold * 2 (since max tr is 2)
            # Normalizing trace to [-1, 1]? No, max trace is 2.
            # "Hot" plaquette if trace is small.
            # Ordered = 2. Disordered = 0.

            # Let's say defect if Trace < 1.0 (arbitrary "cold" cutoff)
            count += jnp.sum(tr < (threshold * 2.0))
            num_plaq += nn**4

    return count / num_plaq

def run_simulation():
    print(f"--- A100 Lattice YM Scan (L={L}^4) ---")
    print("Initializing JAX...")
    # JAX will use GPU if available
    print(f"Devices: {jax.devices()}")

    key = random.PRNGKey(42)

    # Lattice State: (L, L, L, L, 4_dirs, 4_su2_components)
    U = init_lattice(key, (L, L, L, L, 4))
    print(f"Lattice initialized. Shape: {U.shape}")

    results = []

    for beta in BETAS:
        t0 = time.time()
        print(f"Simulating Beta = {beta:.2f}...")

        # We assume a Metropolis update step exists (omitted for brevity in this scaffold)
        # For the scaffolding, we will just measure the INITIAL random state to prove pipeline works.
        # In real run, we insert update loop here.

        # Placeholder Update: Re-randomize slightly (Diffusive noise) to simulate flow
        key, subkey = random.split(key)
        noise = random.normal(subkey, U.shape) * 0.1
        U = U + noise
        U = U / jnp.linalg.norm(U, axis=-1, keepdims=True) # Renormalize to SU(2)

        defect_rho = compute_defect_density(U)
        action_dens = compute_action_density(U, beta)

        # Block until ready
        defect_rho.block_until_ready()
        dt = time.time() - t0

        print(f"  Beta {beta:.2f}: Defect={defect_rho:.4f}, Action={action_dens:.4f} ({dt:.2f}s)")
        results.append((beta, float(defect_rho), float(action_dens)))

    print("\n--- Results ---")
    for b, d, a in results:
        print(f"{b:.2f}, {d:.4f}, {a:.4f}")

if __name__ == "__main__":
    run_simulation()


--- A100 Lattice YM Scan (L=16^4) ---
Initializing JAX...
Devices: [CudaDevice(id=0)]
Lattice initialized. Shape: (16, 16, 16, 16, 4, 4)
Simulating Beta = 1.80...
  Beta 1.80: Defect=0.8043, Action=0.9994 (2.03s)
Simulating Beta = 1.90...
  Beta 1.90: Defect=0.8036, Action=0.9988 (0.00s)
Simulating Beta = 2.00...
  Beta 2.00: Defect=0.8036, Action=0.9992 (0.00s)
Simulating Beta = 2.10...
  Beta 2.10: Defect=0.8040, Action=0.9991 (0.00s)
Simulating Beta = 2.20...
  Beta 2.20: Defect=0.8042, Action=0.9989 (0.00s)
Simulating Beta = 2.30...
  Beta 2.30: Defect=0.8043, Action=0.9992 (0.00s)
Simulating Beta = 2.40...
  Beta 2.40: Defect=0.8045, Action=0.9995 (0.00s)
Simulating Beta = 2.50...
  Beta 2.50: Defect=0.8046, Action=0.9997 (0.00s)
Simulating Beta = 2.60...
  Beta 2.60: Defect=0.8043, Action=0.9998 (0.00s)
Simulating Beta = 2.70...
  Beta 2.70: Defect=0.8044, Action=0.9997 (0.00s)
Simulating Beta = 2.80...
  Beta 2.80: Defect=0.8040, Action=0.9999 (0.00s)
Simulating Beta = 2.90...
 

In [2]:

import os
import time
import jax
import jax.numpy as jnp
from jax import random, jit, lax
import numpy as np

"""
run_defect_scan_gpu_v2.py

A100-Optimized Lattice Yang-Mills Simulation (Full Metropolis).
Objective: Detect the "Spark" (Mass Gap Formation) via Monopole/Vortex Defect Density.

Physics:
- SU(2) Lattice Gauge Theory in 4D.
- Wilson Action S = beta * Sum (1 - 0.5 Tr U_p).
- Metropolis Checkerboard Update.

NOTE: This script is computationally intensive. PROCEED WITH A100.
"""

# Configuration
L = 16
WARMUP = 500   # Sufficient for 16^4
STEPS = 1000   # Measurement steps
BETAS = np.linspace(1.8, 3.0, 13)
METROPOLIS_EPS = 0.2 # Step size for updates

# --- SU(2) Geometry (Same as Scaffold) ---

@jit
def su2_mul(u, v):
    u0, u_vec = u[..., 0], u[..., 1:]
    v0, v_vec = v[..., 0], v[..., 1:]
    res0 = u0*v0 - jnp.sum(u_vec*v_vec, axis=-1)
    res_vec = u0[..., None]*v_vec + v0[..., None]*u_vec + jnp.cross(u_vec, v_vec)
    return jnp.concatenate([res0[..., None], res_vec], axis=-1)

@jit
def su2_dag(u):
    return u * jnp.array([1., -1., -1., -1.])

@jit
def tr_u(u):
    return 2.0 * u[..., 0]

# --- Metropolis Kernel ---

@jit
def compute_staples(U, parity_mask):
    # Calculate sum of staples for links at sites with parity `parity_mask`.
    # Staples for U_mu(x) involve 6 neighbors.
    # This is the most expensive part.
    # U shape: (T, X, Y, Z, 4, 4)
    # Masking is complex in JAX without boolean indexing slowdowns.
    # Efficient approach: Compute staples for ALL sites, then mask.

    total_staple = jnp.zeros_like(U)
    # Placeholder: Full Staple computation is 6 terms per link.
    # For A100 demo, we use a simplified "Cooling" like force or just random drift?
    # No, we promised real physics.

    # Let's implement just ONE staple direction to prove concept,
    # ensuring the User can extend or we can iterate.
    # Full Wilson loop staple is standard.

    return total_staple # TODO: Implement full staple for production run.

# ... Wait, implementing full 4D staple in a single artifact without testing is risky.
# Let's provide the User with the ROBUST SCAFFOLD that runs (from previous step)
# and ask them to run THAT on A100 to map the phase space,
# while acknowledging the Metropolis kernel needs the "Heavy Lifter" update.

# ACTUALLY, simpler idea:
# Random SU(2) update `X` near Identity.
# New Link U' = X * U.
# dS = S_new - S_old.
# Accept if rand < exp(-dS).

@jit
def metropolis_update(key, U, beta):
    # Checkerboard or Global?
    # Global update valid if we update links sequentially? No, violates detailed balance.
    # We must update independent links.
    # But JAX likes large batch arrays.
    # We update ALL links simultaneously with accept/reject?
    # This is "approximate" but works for thermalization if step is small.
    # Strict checkerboard is better.

    key, subkey = random.split(key)

    # 1. Propose Change
    # Random SU(2) matrix X near identity
    raw = random.normal(subkey, U.shape) * METROPOLIS_EPS
    raw = raw.at[..., 0].add(1.0) # Bias to identity
    X = raw / jnp.linalg.norm(raw, axis=-1, keepdims=True)

    U_prop = su2_mul(X, U)

    # 2. Compute Change in Action
    # This requires staples.
    # S_loc = - beta/2 * Tr(U * Staple).
    # Since we lack staples, we can't do exact Metropolis yet.
    # BUT, we can run a "Random Walk" (Infinite Temperature, Beta=0) to test the Defect Density code.
    # Or strict "Cold Start".

    # For the User's A100, they likely want the REAL Thing.
    # I will stick to the Scaffold for now and tell the User:
    # "The A100 script is ready for the connectivity/benchmark test.
    # Once verified, I will drop the 'Staple Kernel' into it."

    return key, U_prop # Just accept everything (Infinite Temp)

# Revert to V1 logic but clean it up for handoff.
pass


In [3]:
import os
import time
import jax
import jax.numpy as jnp
from jax import random, jit, lax, vmap
import numpy as np

"""
run_defect_scan_gpu_production.py

A100-Optimized Lattice Yang-Mills Simulation (Full Metropolis).
Objective: Detect the "Spark" (Mass Gap Formation) via Monopole/Vortex Defect Density.

Physics:
- SU(2) Lattice Gauge Theory in 4D.
- Wilson Action S = beta * Sum (1 - 0.5 Tr U_p).
- Metropolis Checkerboard Update.

NOTE: This script is computationally intensive. PROCEED WITH A100.
"""

# Configuration
L = 16
WARMUP = 1000
STEPS = 200    # Measurements
BETAS = np.linspace(1.8, 3.0, 13)
METROPOLIS_EPS = 0.2
BATCH_SIZE = 1 # Number of simultaneous lattices (keep 1 for max size L)

# --- SU(2) Geometry ---
# Represent SU(2) as 4-vector a = (a0, a1, a2, a3) with norm 1.
# U = a0 I + i vec(a).sigma

@jit
def su2_mul(u, v):
    # u, v shape (..., 4)
    u0, u_vec = u[..., 0], u[..., 1:]
    v0, v_vec = v[..., 0], v[..., 1:]
    res0 = u0*v0 - jnp.sum(u_vec*v_vec, axis=-1)
    # Cross product for a,b in 3D
    res_vec = u0[..., None]*v_vec + v0[..., None]*u_vec + jnp.cross(u_vec, v_vec)
    return jnp.concatenate([res0[..., None], res_vec], axis=-1)

@jit
def su2_dag(u):
    return u * jnp.array([1., -1., -1., -1.])

@jit
def tr_u(u):
    return 2.0 * u[..., 0]

# --- Lattice Geometry Helpers ---

def shift(U, mu, direction):
    # Shift lattice U by 1 step in direction mu
    # axis 0,1,2,3 are T,X,Y,Z
    # direction 1 = forward, -1 = backward
    # We use jnp.roll. A shift of +1 means U(x+mu).
    # roll axis=mu shift=-1 moves index i+1 to i. Correct.
    return jnp.roll(U, -direction, axis=mu)

# --- Staple Computation (The Heavy Lifter) ---

@jit
def compute_staples_at_link(U, mu):
    # For link U_mu(x), sum over 6 staples.
    # Staple_nu_fwd = U_nu(x) U_mu(x+nu) U_nu(x+mu)^dag
    # Staple_nu_bwd = U_nu(x-nu)^dag U_mu(x-nu) U_nu(x+mu-nu)

    total_staple = jnp.zeros_like(U[..., mu, :]) # Shape (L,L,L,L, 4)

    for nu in range(4):
        if nu == mu: continue

        # Forward Staple
        # U_nu(x)
        u_nu = U[..., nu, :]
        # U_mu(x+nu)
        u_mu_fwd = shift(U[..., mu, :], nu, 1)
        # U_nu(x+mu)^dag
        u_nu_shift_dag = su2_dag(shift(U[..., nu, :], mu, 1))

        p_fwd = su2_mul(su2_mul(u_nu, u_mu_fwd), u_nu_shift_dag)
        total_staple += p_fwd

        # Backward Staple
        # U_nu(x-nu)^dag
        u_nu_back_dag = su2_dag(shift(U[..., nu, :], nu, -1))
        # U_mu(x-nu)
        u_mu_back = shift(U[..., mu, :], nu, -1)
        # U_nu(x+mu-nu)
        # shift mu=+1, nu=-1
        u_nu_cross = shift(shift(U[..., nu, :], mu, 1), nu, -1)

        p_bwd = su2_mul(su2_mul(u_nu_back_dag, u_mu_back), u_nu_cross)
        total_staple += p_bwd

    return total_staple

# --- Checkerboard Update ---

@jit
def update_links_checkerboard(key, U, beta, parity):
    # Parity: 0 or 1.
    # Site parity p = (t+x+y+z) % 2.
    # We only update links originating at sites with parity p.
    # Note: U_mu(x) lives on link from x to x+mu.
    # Updating U_mu(x) only depends on staples which use neighbors.
    # Is it safe?
    # Staples use U_nu(x), U_nu(x-nu), U_mu(x+nu), U_mu(x-nu).
    # If x has parity p:
    # x+nu and x-nu have parity 1-p.
    # So staples depend on links at sites (1-p) mostly?
    # BUT U_nu(x) starts at x (parity p).
    # STRICT CHECKERBOARD for gauge fields:
    # A link U_mu(x) acts like a "site" variable? No.
    # Standard approach: Update all links at even sites, then all links at odd sites?
    # Yes, because the Staple for U_mu(x) involves links U_nu(x) (same site) and U_nu(x+mu) (diff site).
    # Actually, link updates are coupled if they share a plaquette.
    # Plaquette involves x, x+mu, x+nu, x+mu+nu.
    # Parities: p, 1-p, 1-p, p.
    # Standard heatbath/metro usually updates all links at site x, but holds site x neighbors fixed.
    # This implies we can update all RED sites' links in parallel, then BLACK sites.
    # Let's assume this standard checkerboard.

    # 1. Create Parity Mask
    grid = jnp.indices(U.shape[:4])
    site_sum = grid[0] + grid[1] + grid[2] + grid[3]
    mask = (site_sum % 2) == parity
    mask = mask[..., None, None] # Expand for (4, 4_su2) dims

    # 2. Staple Calculation (For ALL sites, but we only use masked)
    # Optimization: Calculate staples only for relevant sites?
    # vmap might handle it, but full array ops are cleaner in JAX.

    # We loop over directions mu inside to save memory
    U_new = U

    for mu in range(4):
        staple = compute_staples_at_link(U, mu)

        # 3. Propose Update
        key, subkey = random.split(key)

        # Random step X near identity
        raw = random.normal(subkey, U.shape[:-1] + (4,)) * METROPOLIS_EPS
        raw = raw.at[..., 0].add(1.0)
        X = raw / jnp.linalg.norm(raw, axis=-1, keepdims=True)
        # Only take X component for current mu
        X_mu = X[..., mu, :]

        U_mu = U[..., mu, :]
        U_prop = su2_mul(X_mu, U_mu)

        # 4. Metropolis Test
        # dS = - beta/2 * Tr( (U_new - U_old) * Staple_dag )
        # Actually S = - beta/2 Tr(U P_dag). Maximize Tr(U P_dag).
        # Staple calculated above is sum of products that complete the loop WITHOUT dag.
        # Plaquette = U_mu * Staple_part.
        # So we want to maximize Re Tr (U_mu * Staple_dag).
        # My compute_staples returns the sum of "U_nu ...".
        # Let's verify: P = U_mu * U_nu_fwd * U_mu_fwd_dag * U_nu_dag.
        # So "Staple" = U_nu_fwd * U_mu_fwd_dag * U_nu_dag.
        # And we calculate Tr(U_mu * Staple).
        # Wait, compute_staples logic: P_fwd = U_nu * U_mu_fwd * U_nu_shift_dag.
        # That closes the loop?
        # Loop: x -> x+nu -> x+nu+mu -> x+mu -> x.
        # U_nu(x) * U_mu(x+nu) * U_nu(x+mu)^dag * U_mu(x)^dag.
        # So Staple for U_mu(x) needs to be: U_nu(x) * U_mu(x+nu) * U_nu(x+mu)^dag.
        # Then we multiply by U_mu(x)^dag.
        # My code calculated: P_fwd = U_nu * U_mu_fwd * U_nu_shift_dag. Correct.
        # And we want to maximize Tr(U_mu * Staple_dag)? No.
        # Action S = Sum (1 - 0.5 Tr P).
        # We want to minimize S => Maximize Tr P = Tr(U_mu * Staple_dag).
        # My staple is P_part. U * P_part is the plaquette?
        # P_part corresponds to the "Doorframe" U_nu...
        # So Trace is Tr(U_mu * P_part_dag).

        staple_dag = su2_dag(staple)

        old_tr = tr_u(su2_mul(U_mu, staple_dag))
        new_tr = tr_u(su2_mul(U_prop, staple_dag))

        dS = - (beta / 2.0) * (new_tr - old_tr)

        # Accept if rand < exp(-dS)
        key, sk2 = random.split(key)
        rnd = random.uniform(sk2, dS.shape)
        accept_prob = jnp.exp(-dS)
        accept = rnd < accept_prob

        # Apply mask
        # We assume independent updates for links at different sites?
        # Update link U_mu(x) depends on staples at x.
        # Neighbors of x are x+nu, x-nu.
        # With checkerboard, x is Red. Neighbors are Black.
        # Links at Black sites are NOT changing in this pass.
        # But U_mu(x) lives on link x->x+mu.
        # Does staple for U_mu(x) involve OTHER links starting at Red sites?
        # Staple: U_nu(x) (Red start), U_mu(x+nu) (Black start), U_nu(x+mu) (Black start).
        # ONLY U_nu(x) is red-start.
        # So U_mu(x) update depends on U_nu(x).
        # If we update U_mu and U_nu simultaneously, we have a conflict?
        # YES. They share plaquettes.
        # We must update Direction-by-Direction serially or be careful.
        # To be safe: Serial loop over directions mu inside the kernel.
        # I did `for mu in range(4)`.
        # Inside this loop, `U` is updated?
        # JAX pure func: we must accumulate updates.
        # `U_new` tracks changes.

        # Masking
        # `mask` is shape (L,L,L,L,1,1).
        # `accept` is shape (L,L,L,L).
        do_update = accept[..., None] * mask[..., mu, :] # logical AND

        U_mu_new = jnp.where(do_update, U_prop, U_mu)
        U_new = U_new.at[..., mu, :].set(U_mu_new)

    return key, U_new

# --- Observables ---

@jit
def compute_defect_density(U, threshold=0.5):
    # Same as before
    count = 0.0
    nn = U.shape[0]
    num_plaq = 0.0

    # We can batch this loop?
    # Manual loop ok for observability
    for mu in range(4):
        for nu in range(mu+1, 4):
            u_mu = U[..., mu, :]
            u_nu_shift = shift(U[..., nu, :], mu, 1)
            u_mu_shift_dag = su2_dag(shift(U[..., mu, :], nu, 1))
            u_nu_dag = su2_dag(U[..., nu, :])

            plt = su2_mul(su2_mul(u_mu, u_nu_shift), su2_mul(u_mu_shift_dag, u_nu_dag))
            tr = tr_u(plt)

            # Defect if trace < threshold (max 2)
            # Threshold 0.5 means < 1.0
            count += jnp.sum(tr < (threshold * 2.0))
            num_plaq += nn**4

    return count / num_plaq

def run_simulation_production():
    print(f"--- A100 Lattice YM PRODUCTION (L={L}^4) ---")
    print(f"JAX Devices: {jax.devices()}")

    key = random.PRNGKey(42)
    U = random.normal(key, (L, L, L, L, 4, 4))
    U = U / jnp.linalg.norm(U, axis=-1, keepdims=True)

    results = []

    # Pre-compile
    print("Compiling Kernels (can take 30s)...")
    _ = update_links_checkerboard(key, U, 2.0, 0)
    _ = compute_defect_density(U)
    print("Compilation Done.")

    for beta in BETAS:
        t0 = time.time()
        print(f"Simulating Beta = {beta:.2f}...")

        # Thermalize
        # We need a Python loop for steps because scanning large arrays kills memory?
        # Actually lax.scan is better.
        # But let's do explicit loop for observability.

        # Annealing: Reset or Continue?
        # Continue from previous Beta (Hysteresis check?)
        # Or Hot Start?
        # Let's Hot Start every Beta to avoid getting stuck?
        # Or just thermalize long enough.
        # Hot Start for independent samples.
        key, rkey = random.split(key)
        U = random.normal(rkey, (L, L, L, L, 4, 4))
        U = U / jnp.linalg.norm(U, axis=-1, keepdims=True)

        # WARMUP
        for _ in range(WARMUP):
            key, U = update_links_checkerboard(key, U, beta, 0) # Even
            key, U = update_links_checkerboard(key, U, beta, 1) # Odd

        # MEASURE
        defect_acc = 0.0
        for _ in range(STEPS):
            key, U = update_links_checkerboard(key, U, beta, 0)
            key, U = update_links_checkerboard(key, U, beta, 1)

            # Measure every 10 steps to reduce correlation?
            # For A100, we can measure every step.
            dd = compute_defect_density(U)
            defect_acc += dd

        defect_avg = defect_acc / STEPS
        dt = time.time() - t0

        print(f"  Beta {beta:.2f}: Defect={defect_avg:.4f} ({dt:.2f}s)")
        results.append((beta, float(defect_avg)))

    print("\n--- Results ---")
    for b, d in results:
        print(f"{b:.2f}, {d:.4f}")

if __name__ == "__main__":
    run_simulation_production()

--- A100 Lattice YM PRODUCTION (L=16^4) ---
JAX Devices: [CudaDevice(id=0)]
Compiling Kernels (can take 30s)...


TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function compute_staples_at_link at /tmp/ipython-input-3218453846.py:64 for jit. This concrete value was not available in Python because it depends on the value of the argument mu.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

In [5]:

import os
import time
import jax
import jax.numpy as jnp
from jax import random, jit, lax, vmap
import numpy as np

"""
run_defect_scan_gpu_production.py

A100-Optimized Lattice Yang-Mills Simulation (Full Metropolis).
Objective: Detect the "Spark" (Mass Gap Formation) via Monopole/Vortex Defect Density.

Physics:
- SU(2) Lattice Gauge Theory in 4D.
- Wilson Action S = beta * Sum (1 - 0.5 Tr U_p).
- Metropolis Checkerboard Update.

NOTE: This script is computationally intensive. PROCEED WITH A100.
"""

# Configuration
L = 16
WARMUP = 1000
STEPS = 200    # Measurements
BETAS = np.linspace(1.8, 3.0, 13)
METROPOLIS_EPS = 0.2
BATCH_SIZE = 1 # Number of simultaneous lattices (keep 1 for max size L)

# --- SU(2) Geometry ---
# Represent SU(2) as 4-vector a = (a0, a1, a2, a3) with norm 1.
# U = a0 I + i vec(a).sigma

@jit
def su2_mul(u, v):
    # u, v shape (..., 4)
    u0, u_vec = u[..., 0], u[..., 1:]
    v0, v_vec = v[..., 0], v[..., 1:]
    res0 = u0*v0 - jnp.sum(u_vec*v_vec, axis=-1)
    # Cross product for a,b in 3D
    res_vec = u0[..., None]*v_vec + v0[..., None]*u_vec + jnp.cross(u_vec, v_vec)
    return jnp.concatenate([res0[..., None], res_vec], axis=-1)

@jit
def su2_dag(u):
    return u * jnp.array([1., -1., -1., -1.])

@jit
def tr_u(u):
    return 2.0 * u[..., 0]

# --- Lattice Geometry Helpers ---

def shift(U, mu, direction):
    # Shift lattice U by 1 step in direction mu
    # axis 0,1,2,3 are T,X,Y,Z
    # direction 1 = forward, -1 = backward
    # We use jnp.roll. A shift of +1 means U(x+mu).
    # roll axis=mu shift=-1 moves index i+1 to i. Correct.
    return jnp.roll(U, -direction, axis=mu)

# --- Staple Computation (The Heavy Lifter) ---

# --- Staple Computation (The Heavy Lifter) ---

# NOTE: We do NOT jit this helper explicitly because it is called inside a loop
# in the main kernel. We want JAX to unroll the loop with concrete 'mu'.
# If we jit this, 'mu' becomes a Tracer and causes 'if nu == mu' to fail.
def compute_staples_at_link(U, mu):
    # For link U_mu(x), sum over 6 staples.
    # Staple_nu_fwd = U_nu(x) U_mu(x+nu) U_nu(x+mu)^dag
    # Staple_nu_bwd = U_nu(x-nu)^dag U_mu(x-nu) U_nu(x+mu-nu)

    total_staple = jnp.zeros_like(U[..., mu, :]) # Shape (L,L,L,L, 4)

    for nu in range(4):
        if nu == mu: continue

        # Forward Staple
        # U_nu(x)
        u_nu = U[..., nu, :]
        # U_mu(x+nu)
        u_mu_fwd = shift(U[..., mu, :], nu, 1)
        # U_nu(x+mu)^dag
        u_nu_shift_dag = su2_dag(shift(U[..., nu, :], mu, 1))

        p_fwd = su2_mul(su2_mul(u_nu, u_mu_fwd), u_nu_shift_dag)
        total_staple += p_fwd

        # Backward Staple
        # U_nu(x-nu)^dag
        u_nu_back_dag = su2_dag(shift(U[..., nu, :], nu, -1))
        # U_mu(x-nu)
        u_mu_back = shift(U[..., mu, :], nu, -1)
        # U_nu(x+mu-nu)
        # shift mu=+1, nu=-1
        u_nu_cross = shift(shift(U[..., nu, :], mu, 1), nu, -1)

        p_bwd = su2_mul(su2_mul(u_nu_back_dag, u_mu_back), u_nu_cross)
        total_staple += p_bwd

    return total_staple

# --- Checkerboard Update ---

@jit
def update_links_checkerboard(key, U, beta, parity):
    # Parity: 0 or 1.
    # Site parity p = (t+x+y+z) % 2.
    # We only update links originating at sites with parity p.
    # Note: U_mu(x) lives on link from x to x+mu.
    # Updating U_mu(x) only depends on staples which use neighbors.
    # Is it safe?
    # Staples use U_nu(x), U_nu(x-nu), U_mu(x+nu), U_mu(x-nu).
    # If x has parity p:
    # x+nu and x-nu have parity 1-p.
    # So staples depend on links at sites (1-p) mostly?
    # BUT U_nu(x) starts at x (parity p).
    # STRICT CHECKERBOARD for gauge fields:
    # A link U_mu(x) acts like a "site" variable? No.
    # Standard approach: Update all links at even sites, then all links at odd sites?
    # Yes, because the Staple for U_mu(x) involves links U_nu(x) (same site) and U_nu(x+mu) (diff site).
    # Actually, link updates are coupled if they share a plaquette.
    # Plaquette involves x, x+mu, x+nu, x+mu+nu.
    # Parities: p, 1-p, 1-p, p.
    # Standard heatbath/metro usually updates all links at site x, but holds site x neighbors fixed.
    # This implies we can update all RED sites' links in parallel, then BLACK sites.
    # Let's assume this standard checkerboard.

    # 1. Create Parity Mask
    grid = jnp.indices(U.shape[:4])
    site_sum = grid[0] + grid[1] + grid[2] + grid[3]
    mask = (site_sum % 2) == parity
    mask = mask[..., None, None] # Expand for (4, 4_su2) dims

    # 2. Staple Calculation (For ALL sites, but we only use masked)
    # Optimization: Calculate staples only for relevant sites?
    # vmap might handle it, but full array ops are cleaner in JAX.

    # We loop over directions mu inside to save memory
    U_new = U

    for mu in range(4):
        staple = compute_staples_at_link(U, mu)

        # 3. Propose Update
        key, subkey = random.split(key)

        # Random step X near identity
        raw = random.normal(subkey, U.shape[:-1] + (4,)) * METROPOLIS_EPS
        raw = raw.at[..., 0].add(1.0)
        X = raw / jnp.linalg.norm(raw, axis=-1, keepdims=True)
        # Only take X component for current mu
        X_mu = X[..., mu, :]

        U_mu = U[..., mu, :]
        U_prop = su2_mul(X_mu, U_mu)

        # 4. Metropolis Test
        # dS = - beta/2 * Tr( (U_new - U_old) * Staple_dag )
        # Actually S = - beta/2 Tr(U P_dag). Maximize Tr(U P_dag).
        # Staple calculated above is sum of products that complete the loop WITHOUT dag.
        # Plaquette = U_mu * Staple_part.
        # So we want to maximize Re Tr (U_mu * Staple_dag).
        # My compute_staples returns the sum of "U_nu ...".
        # Let's verify: P = U_mu * U_nu_fwd * U_mu_fwd_dag * U_nu_dag.
        # So "Staple" = U_nu_fwd * U_mu_fwd_dag * U_nu_dag.
        # And we calculate Tr(U_mu * Staple).
        # Wait, compute_staples logic: P_fwd = U_nu * U_mu_fwd * U_nu_shift_dag.
        # That closes the loop?
        # Loop: x -> x+nu -> x+nu+mu -> x+mu -> x.
        # U_nu(x) * U_mu(x+nu) * U_nu(x+mu)^dag * U_mu(x)^dag.
        # So Staple for U_mu(x) needs to be: U_nu(x) * U_mu(x+nu) * U_nu(x+mu)^dag.
        # Then we multiply by U_mu(x)^dag.
        # My code calculated: P_fwd = U_nu * U_mu_fwd * U_nu_shift_dag. Correct.
        # And we want to maximize Tr(U_mu * Staple_dag)? No.
        # Action S = Sum (1 - 0.5 Tr P).
        # We want to minimize S => Maximize Tr P = Tr(U_mu * Staple_dag).
        # My staple is P_part. U * P_part is the plaquette?
        # P_part corresponds to the "Doorframe" U_nu...
        # So Trace is Tr(U_mu * P_part_dag).

        staple_dag = su2_dag(staple)

        old_tr = tr_u(su2_mul(U_mu, staple_dag))
        new_tr = tr_u(su2_mul(U_prop, staple_dag))

        dS = - (beta / 2.0) * (new_tr - old_tr)

        # Accept if rand < exp(-dS)
        key, sk2 = random.split(key)
        rnd = random.uniform(sk2, dS.shape)
        accept_prob = jnp.exp(-dS)
        accept = rnd < accept_prob

        # Apply mask
        # We assume independent updates for links at different sites?
        # Update link U_mu(x) depends on staples at x.
        # Neighbors of x are x+nu, x-nu.
        # With checkerboard, x is Red. Neighbors are Black.
        # Links at Black sites are NOT changing in this pass.
        # But U_mu(x) lives on link x->x+mu.
        # Does staple for U_mu(x) involve OTHER links starting at Red sites?
        # Staple: U_nu(x) (Red start), U_mu(x+nu) (Black start), U_nu(x+mu) (Black start).
        # ONLY U_nu(x) is red-start.
        # So U_mu(x) update depends on U_nu(x).
        # If we update U_mu and U_nu simultaneously, we have a conflict?
        # YES. They share plaquettes.
        # We must update Direction-by-Direction serially or be careful.
        # To be safe: Serial loop over directions mu inside the kernel.
        # I did `for mu in range(4)`.
        # Inside this loop, `U` is updated?
        # JAX pure func: we must accumulate updates.
        # `U_new` tracks changes.

        # Masking
        # `mask` is shape (L,L,L,L,1,1).
        # `accept` is shape (L,L,L,L).
        do_update = accept[..., None] * mask[..., mu, :] # logical AND

        U_mu_new = jnp.where(do_update, U_prop, U_mu)
        U_new = U_new.at[..., mu, :].set(U_mu_new)

    return key, U_new

# --- Observables ---

@jit
def compute_defect_density(U, threshold=0.5):
    # Same as before
    count = 0.0
    nn = U.shape[0]
    num_plaq = 0.0

    # We can batch this loop?
    # Manual loop ok for observability
    for mu in range(4):
        for nu in range(mu+1, 4):
            u_mu = U[..., mu, :]
            u_nu_shift = shift(U[..., nu, :], mu, 1)
            u_mu_shift_dag = su2_dag(shift(U[..., mu, :], nu, 1))
            u_nu_dag = su2_dag(U[..., nu, :])

            plt = su2_mul(su2_mul(u_mu, u_nu_shift), su2_mul(u_mu_shift_dag, u_nu_dag))
            tr = tr_u(plt)

            # Defect if trace < threshold (max 2)
            # Threshold 0.5 means < 1.0
            count += jnp.sum(tr < (threshold * 2.0))
            num_plaq += nn**4

    return count / num_plaq

def run_simulation_production():
    print(f"--- A100 Lattice YM PRODUCTION (L={L}^4) ---")
    print(f"JAX Devices: {jax.devices()}")

    key = random.PRNGKey(42)
    U = random.normal(key, (L, L, L, L, 4, 4))
    U = U / jnp.linalg.norm(U, axis=-1, keepdims=True)

    results = []

    # Pre-compile
    print("Compiling Kernels (can take 30s)...")
    _ = update_links_checkerboard(key, U, 2.0, 0)
    _ = compute_defect_density(U)
    print("Compilation Done.")

    for beta in BETAS:
        t0 = time.time()
        print(f"Simulating Beta = {beta:.2f}...")

        # Thermalize
        # We need a Python loop for steps because scanning large arrays kills memory?
        # Actually lax.scan is better.
        # But let's do explicit loop for observability.

        # Annealing: Reset or Continue?
        # Continue from previous Beta (Hysteresis check?)
        # Or Hot Start?
        # Let's Hot Start every Beta to avoid getting stuck?
        # Or just thermalize long enough.
        # Hot Start for independent samples.
        key, rkey = random.split(key)
        U = random.normal(rkey, (L, L, L, L, 4, 4))
        U = U / jnp.linalg.norm(U, axis=-1, keepdims=True)

        # WARMUP
        for _ in range(WARMUP):
            key, U = update_links_checkerboard(key, U, beta, 0) # Even
            key, U = update_links_checkerboard(key, U, beta, 1) # Odd

        # MEASURE
        defect_acc = 0.0
        for _ in range(STEPS):
            key, U = update_links_checkerboard(key, U, beta, 0)
            key, U = update_links_checkerboard(key, U, beta, 1)

            # Measure every 10 steps to reduce correlation?
            # For A100, we can measure every step.
            dd = compute_defect_density(U)
            defect_acc += dd

        defect_avg = defect_acc / STEPS
        dt = time.time() - t0

        print(f"  Beta {beta:.2f}: Defect={defect_avg:.4f} ({dt:.2f}s)")
        results.append((beta, float(defect_avg)))

    print("\n--- Results ---")
    for b, d in results:
        print(f"{b:.2f}, {d:.4f}")

if __name__ == "__main__":
    run_simulation_production()


--- A100 Lattice YM PRODUCTION (L=16^4) ---
JAX Devices: [CudaDevice(id=0)]
Compiling Kernels (can take 30s)...
Compilation Done.
Simulating Beta = 1.80...
  Beta 1.80: Defect=0.4812 (10.82s)
Simulating Beta = 1.90...
  Beta 1.90: Defect=0.4515 (1.53s)
Simulating Beta = 2.00...
  Beta 2.00: Defect=0.4203 (1.53s)
Simulating Beta = 2.10...
  Beta 2.10: Defect=0.3852 (1.53s)
Simulating Beta = 2.20...
  Beta 2.20: Defect=0.3457 (1.53s)
Simulating Beta = 2.30...
  Beta 2.30: Defect=0.3068 (1.53s)
Simulating Beta = 2.40...
  Beta 2.40: Defect=0.2727 (1.53s)
Simulating Beta = 2.50...
  Beta 2.50: Defect=0.2442 (1.53s)
Simulating Beta = 2.60...
  Beta 2.60: Defect=0.2214 (1.53s)
Simulating Beta = 2.70...
  Beta 2.70: Defect=0.2009 (1.53s)
Simulating Beta = 2.80...
  Beta 2.80: Defect=0.1825 (1.53s)
Simulating Beta = 2.90...
  Beta 2.90: Defect=0.1665 (1.53s)
Simulating Beta = 3.00...
  Beta 3.00: Defect=0.1518 (1.53s)

--- Results ---
1.80, 0.4812
1.90, 0.4515
2.00, 0.4203
2.10, 0.3852
2.20, 0